# W04D05 — SHAP Values: Explainability Dasar

**ML Engineer Journey | Phase 1 — ML Fundamentals | 3 Apr 2026**

---

## Tujuan Hari Ini

1. Memahami **mengapa feature importance biasa tidak cukup**
2. Memahami cara kerja SHAP secara intuitif dan matematis
3. Membuat **summary plot** dan **waterfall plot**
4. Menginterpretasi hasil untuk **3 prediksi individual berbeda**

---

## Catatan Penting dari Code Review

> ⚠️ **Bagian ini wajib dibaca sebelum mulai coding.** Ini dokumentasi dari kesalahan yang terjadi hari ini — supaya tidak terulang.

### Kesalahan #1 — Salah mendefinisikan "prediksi yang salah"
`np.argmin(proba)` mencari sample yang paling *confident Malignant* — bukan yang *salah prediksi*.  
**Confident ≠ Wrong.** Ini dua konsep yang berbeda.

```python
# ❌ SALAH — ini mencari confident Malignant, bukan mispredicted
idx_wrong = np.argmin(proba)

# ✅ BENAR — ini mencari yang benar-benar salah diprediksi
wrong_indices = np.where(model.predict(X_test) != y_test.values)[0]
idx_wrong = wrong_indices[np.argmax(np.abs(proba[wrong_indices] - 0.5))]
```

### Kesalahan #2 — Waterfall dibuat untuk kasus yang salah
Waterfall diminta untuk **prediksi yang salah** (untuk analisa blind spot). Tapi waterfall malah dibuat untuk `High Confidence Benign`.  
Plot yang dibuat tidak menjawab pertanyaan yang diminta.

### Kesalahan #3 — Borderline salah dikategorikan
Prob **0.6944** bukan borderline. Borderline = mendekati **0.50**.  
`np.argmin(np.abs(proba - 0.5))` harus menghasilkan prob sekitar 0.50–0.52.

### Kesalahan #4 — Definisi `shap_values[i][j]` terlalu dangkal
Bukan "matriks numpy yang dihasilkan XGBoost" — itu deskripsi struktur, bukan makna.  
**Definisi yang benar:** kontribusi marginal fitur ke-j terhadap prediksi sample ke-i, dinyatakan sebagai deviasi dari base_value.

### Kesalahan #5 — Jawaban Q3 melompat ke "debugging model"
Pertanyaan: *fitur mana yang paling efisien diubah?*  
`worst_texture` adalah blind spot konseptual. Tapi untuk *membalik prediksi pada Index 84*, yang paling efisien adalah **`worst_perimeter`** karena itu penahan terbesar yang sudah ada (-1.0559).


## Setup dan Import

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Output directory
OUTPUT_DIR = "./outputs/shap_values"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup selesai.")
print(f"SHAP version: {shap.__version__}")


## 1. Load Data

Menggunakan breast cancer dataset dari sklearn.
- **Target 0** = Malignant (ganas)
- **Target 1** = Benign (jinak)
- 30 fitur numerik, 569 samples


In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# PENTING: reset index agar sejalan dengan numpy array dari SHAP
# Tanpa ini, iloc dan numpy indexing bisa tidak sinkron
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"Train size : {len(X_train)}")
print(f"Test size  : {len(X_test)}")
print(f"Features   : {X.shape[1]}")
print(f"Class dist : Malignant={sum(y==0)}, Benign={sum(y==1)}")


## 2. Train Model

XGBoost dengan parameter dasar. Ini bukan tuning session — model cukup baik untuk analisa SHAP.


In [ ]:
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric="logloss",
    verbosity=0
)
model.fit(X_train, y_train)

acc = model.score(X_test, y_test)
print(f"Test accuracy: {acc:.4f}")


## 3. Membuat SHAP Explainer

### Mengapa TreeExplainer?

Ada beberapa jenis explainer di SHAP:
- `KernelExplainer` — model-agnostic, bekerja pada semua model, tapi **sangat lambat** (sampling-based)
- `TreeExplainer` — **dioptimasi khusus untuk tree-based model** (XGBoost, RandomForest, LightGBM). Mengeksploitasi struktur pohon secara langsung: kompleksitas O(TLD²) di mana T=pohon, L=daun, D=kedalaman
- `LinearExplainer` — untuk linear model

### Apa itu `shap_values[i][j]`?

Bukan sekedar "isi matriks numpy" — ini adalah:

> **Kontribusi marginal fitur ke-j terhadap prediksi sample ke-i, dinyatakan sebagai deviasi dari base_value.**

Formula lengkap:
```
prediksi(x) = base_value + shap_values[i][0] + shap_values[i][1] + ... + shap_values[i][n]
```


In [ ]:
# TreeExplainer mengeksploitasi struktur pohon langsung — jauh lebih cepat dari KernelExplainer
explainer = shap.TreeExplainer(model)

# Hitung SHAP values untuk seluruh test set
# Hasilnya matrix: (n_samples, n_features)
# Setiap nilai = kontribusi marginal fitur j terhadap prediksi sample i
shap_values = explainer.shap_values(X_test)

print(f"shap_values shape  : {shap_values.shape}")
print(f"Base value         : {explainer.expected_value:.4f}")
print()
print("Verifikasi additivitas (sample 0):")
print(f"  base_value       : {explainer.expected_value:.4f}")
print(f"  sum(shap_values) : {shap_values[0].sum():.4f}")
print(f"  base + shap sum  : {explainer.expected_value + shap_values[0].sum():.4f}")
proba_check = model.predict_proba(X_test)[:, 1]
print(f"  actual prob[0]   : {proba_check[0]:.4f}")
print()
print("(base + shap sum) harus sama dengan actual prob — ini membuktikan additivitas SHAP")


## 4. Summary Plot

### Cara Membaca Summary Plot

Summary plot menggabungkan dua informasi:
1. **Sumbu X** = magnitude SHAP value (seberapa besar pengaruhnya)
2. **Warna titik** = nilai fitur asli (merah = tinggi, biru = rendah)

**Contoh interpretasi:**
- Jika titik **merah** berada di **kanan** (+): nilai tinggi fitur itu mendorong ke Benign
- Jika titik **merah** berada di **kiri** (-): nilai tinggi fitur itu mendorong ke Malignant


In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test,
    show=False
)
plt.title("SHAP Summary Plot — Breast Cancer Dataset", pad=15)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "shap_summary_plot.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()  # PENTING: cegah memory leak
print(f"Tersimpan: {save_path}")


### Interpretasi Summary Plot

Tulis observasi kamu di sini setelah melihat plot:

- **Fitur paling berpengaruh:** worst texture dan worst perimeter (distribusi SHAP terlebar)
- **worst perimeter tinggi** (titik merah, kiri) → mendorong ke **Malignant**
- **worst perimeter rendah** (titik biru, kanan) → mendorong ke **Benign**
- **worst texture**: nilai tinggi dengan SHAP rendah → mendorong ke Malignant; nilai rendah → mendorong ke Benign


## 5. Analisa 3 Prediksi Individual

Tiga jenis kasus yang berbeda:
1. **High confidence benign** — model sangat yakin, dan benar
2. **Borderline** — model ragu-ragu (prob ~0.5)
3. **Mispredicted** — model **salah** prediksi

---

### ⚠️ Catatan Penting: Cara Mencari Kasus yang Benar

```python
# ❌ KESALAHAN SEBELUMNYA — ini bukan "prediksi salah"
idx_wrong = np.argmin(proba)  # ini mencari confident Malignant, bukan mispredicted!

# ✅ BENAR — cari yang benar-benar salah diprediksi
wrong_indices = np.where(model.predict(X_test) != y_test.values)[0]
# Dari yang salah, pilih yang paling confident (paling jauh dari 0.5)
idx_wrong = wrong_indices[np.argmax(np.abs(proba[wrong_indices] - 0.5))]
```

**Confident ≠ Wrong. Ini dua konsep yang sepenuhnya berbeda.**


In [ ]:
proba = model.predict_proba(X_test)[:, 1]
preds = model.predict(X_test)

# ── Kasus 1: High Confidence Benign ──────────────────────────────────────────
idx_high_benign = np.argmax(proba)

# ── Kasus 2: Borderline ──────────────────────────────────────────────────────
idx_borderline = np.argmin(np.abs(proba - 0.5))

# ── Kasus 3: Mispredicted ─────────────────────────────────────────────────────
# PENTING: ini cara yang BENAR — bukan np.argmin(proba)
wrong_indices = np.where(preds != y_test.values)[0]
if len(wrong_indices) > 0:
    # Pilih yang paling confident di antara yang salah
    idx_wrong = wrong_indices[np.argmax(np.abs(proba[wrong_indices] - 0.5))]
else:
    print("Model sempurna di test set — tidak ada mispredicted case!")
    idx_wrong = None

cases = {
    "High Confidence Benign": idx_high_benign,
    "Borderline":             idx_borderline,
    "Mispredicted":           idx_wrong,
}

print("Index kasus yang dipilih:")
for name, idx in cases.items():
    if idx is not None:
        actual = "Benign" if y_test.iloc[idx] == 1 else "Malignant"
        predicted = "Benign" if preds[idx] == 1 else "Malignant"
        correct = "BENAR" if y_test.iloc[idx] == preds[idx] else "SALAH"
        print(f"  {name:<25} idx={idx:<4} actual={actual:<10} predicted={predicted:<10} [{correct}] prob={proba[idx]:.4f}")


In [ ]:
def print_shap_detail(idx, case_name):
    """
    Cetak detail SHAP untuk satu sample.

    Args:
        idx (int): Index sample di X_test.
        case_name (str): Label kasus untuk display.
    """
    actual_label = "Benign" if y_test.iloc[idx] == 1 else "Malignant"
    predicted_label = "Benign" if preds[idx] == 1 else "Malignant"
    correct = "BENAR" if y_test.iloc[idx] == preds[idx] else "SALAH ❌"

    print(f"{'='*60}")
    print(f"Kasus  : {case_name} (Index: {idx})")
    print(f"  Aktual         : {actual_label}")
    print(f"  Prediksi       : {predicted_label} [{correct}]")
    print(f"  Prob Benign    : {proba[idx]:.4f}")
    print(f"  Base Value     : {explainer.expected_value:.4f}")

    shap_row = shap_values[idx]
    top_indices = np.argsort(np.abs(shap_row))[::-1][:5]

    print(f"  Top 5 Fitur Berpengaruh:")
    for rank, fi in enumerate(top_indices, 1):
        direction = "Mendorong ke Benign    (+)" if shap_row[fi] > 0 else "Mendorong ke Malignant (-)"
        print(f"    {rank}. {X_test.columns[fi]:<30} | {direction} | "
              f"SHAP: {shap_row[fi]:+.4f} | Nilai: {X_test.iloc[idx, fi]:.2f}")

for name, idx in cases.items():
    if idx is not None:
        print_shap_detail(idx, name)


### Interpretasi per Kasus

**Kasus 1 — High Confidence Benign:**
> Model memprediksi Benign dengan sangat yakin (prob ~0.999) karena hampir semua fitur teratas sepakat mendorong ke arah yang sama — worst texture, worst perimeter, worst area semuanya memberikan SHAP positif besar.

**Kasus 2 — Borderline:**
> Model ragu karena ada pertarungan antara fitur yang berlawanan arah. Beberapa fitur mendorong ke Benign, beberapa ke Malignant, sehingga probabilitas mendekati 0.5.

**Kasus 3 — Mispredicted:**
> Ini yang paling penting untuk dipelajari. Identifikasi fitur "penipu" — fitur yang memberikan sinyal besar ke arah yang salah. Biasanya satu fitur dengan SHAP magnitude besar tapi arahnya berlawanan dengan label aktual.

**Pertanyaan kritis untuk kasus mispredicted:**
Jika kamu ingin membalik prediksi model agar benar, fitur mana yang paling efisien diubah?  
→ Bukan fitur yang "menipu" (yang sudah terlanjur mendorong salah) — tapi **penahan terbesar** yang sudah ada di arah yang benar. Perkuat penahan itu.


## 6. Waterfall Plot — Analisa Prediksi yang Salah

Waterfall plot diminta untuk **prediksi yang salah** karena:
- Kita ingin tahu **mengapa** model salah pada kasus itu
- Kita bisa identifikasi fitur "blind spot" yang menyesatkan model
- Ini informasi yang tidak bisa didapat dari summary plot (yang sifatnya global)

### ⚠️ Kesalahan Sebelumnya
Waterfall sebelumnya dibuat untuk `High Confidence Benign` — padahal yang diminta adalah untuk **mispredicted case**. Plot yang benar tapi dibuat untuk kasus yang salah tidak menjawab pertanyaan apapun.


In [ ]:
def plot_waterfall(idx, filename, title_suffix=""):
    """
    Membuat dan menyimpan waterfall plot untuk satu sample.

    Args:
        idx (int): Index sample di X_test.
        filename (str): Nama file output PNG.
        title_suffix (str): Tambahan judul untuk konteks.
    """
    shap_explanation = shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_test.iloc[idx].values,
        feature_names=X_test.columns.tolist()
    )

    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap_explanation, show=False)

    actual = "Benign" if y_test.iloc[idx] == 1 else "Malignant"
    predicted = "Benign" if preds[idx] == 1 else "Malignant"
    plt.title(f"Waterfall Plot — Index {idx} | Aktual: {actual} | Prediksi: {predicted}{title_suffix}", pad=15)
    plt.tight_layout()

    save_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Tersimpan: {save_path}")

# Waterfall untuk mispredicted case — INI yang diminta di task
if idx_wrong is not None:
    plot_waterfall(idx_wrong, "waterfall_mispredicted.png", " [SALAH PREDIKSI]")
else:
    print("Tidak ada mispredicted case untuk divisualisasi.")


### Analisa Waterfall — Prediksi yang Salah

Setelah melihat waterfall plot, identifikasi:

**Fitur yang menyesatkan model ("blind spot"):**
- Fitur dengan SHAP magnitude besar tapi arahnya berlawanan dengan label aktual
- Ini adalah titik lemah model — ada kondisi di mana model over-trust fitur tertentu

**Fitur penahan yang kalah:**
- Fitur yang sudah berada di arah yang benar tapi tidak cukup kuat
- Jika ingin "memperbaiki" prediksi ini, perkuat fitur-fitur penahan ini

**Pelajaran penting:**
Model machine learning tidak "tahu" tentang penyakit — ia hanya belajar pola statistik dari training data. Ketika distribusi fitur di dunia nyata tidak sesuai pola training, model bisa salah dengan sangat percaya diri.


## 7. Bonus — Waterfall untuk High Confidence Benign

In [ ]:
# Plot ini valid sebagai pembanding — bukan sebagai jawaban task utama
plot_waterfall(idx_high_benign, "waterfall_high_benign.png", " [HIGH CONFIDENCE]")


## Ringkasan Akhir

### Yang Dipelajari Hari Ini

| Konsep | Inti |
|--------|------|
| SHAP value | Kontribusi marginal fitur j terhadap prediksi sample i, deviasi dari base_value |
| Base value | Rata-rata prediksi model di training set — titik netral |
| TreeExplainer | Dioptimasi untuk tree model, O(TLD²), jauh lebih cepat dari KernelExplainer |
| Summary plot | Gambaran global: magnitude + arah + distribusi nilai fitur |
| Waterfall plot | Detail satu prediksi: siapa yang mendorong ke mana |
| Additivitas | base_value + sum(shap_values) = prediksi akhir, selalu |

### Koreksi yang Harus Diingat

1. `np.argmin(proba)` ≠ prediksi yang salah
2. Waterfall untuk task "analisa blind spot" harus dibuat untuk **mispredicted case**
3. Borderline = prob mendekati **0.5**, bukan hanya "kurang confident"
4. `shap_values[i][j]` = **kontribusi marginal** — bukan deskripsi struktur matriks
5. Untuk membalik prediksi: perkuat **penahan terbesar**, bukan ubah "fitur penipu"

---

*W04D05 — ML Engineer Journey | github.com/lolivampire/ML-Project*
